# SDG 3 Indicator Text Classification — End-to-End Pipeline

This notebook runs the full pipeline used in our report: load → EDA → preprocess → eight progressive experiments → final ensemble → submission CSV.

**Reproducibility contract.** Every cell calls into `src/sdgtext/`. There is no logic in the notebook that doesn't also exist in the package; that means CI tests and the CLI cover the same code paths a grader exercises here.

**Compute.** CPU-only. The heaviest step is the Sentence-BERT pass (Experiments 6–8), which encodes ~3K documents in ~3–5 minutes on a Colab CPU runtime. All other experiments complete in seconds.

**Primary metric.** Hamming Loss (lower = better). Secondary: micro-F1, macro-F1, samples-F1, subset accuracy, LRAP.

## 0. Setup (Colab)

Uncomment the next cell only on Colab. Locally, run `pip install -e .` from the repo root once and skip it.

In [ ]:
# !git clone https://github.com/<your-org>/SDG-Indicator-Text.git
# %cd SDG-Indicator-Text
# !pip install -q -r requirements.txt
# !pip install -q -e .

In [ ]:
import json, os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make src/ importable when running from notebooks/ without installing.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_context('notebook'); sns.set_style('whitegrid')

from sdgtext.utils.seeding import seed_everything
from sdgtext.utils.config import load_config
from sdgtext.utils.logging import get_logger
from sdgtext.data.load import load_devex
from sdgtext.data.preprocess import PreprocessConfig, normalize_corpus
from sdgtext.eval import eda as eda_mod
from sdgtext.models.pipeline import run_experiment

seed_everything(42)
log = get_logger('notebook')
log.info('Environment ready.')

## 1. Load the Devex SDG-3 dataset

Place `Devex_train.csv` and `Devex_test_questions.csv` under `data/raw/` before running this cell. The loader normalizes the schema (joins title + description into `__text__`, autodetects label columns, surfaces data-quality issues).

In [ ]:
cfg = load_config('default.yaml')
train_ds, issues = load_devex(
    path=cfg['data']['raw_train'],
    text_columns=cfg['data']['text_columns'],
    id_column=cfg['data'].get('id_column'),
    meta_columns=cfg['data'].get('meta_columns', []),
    label_format=cfg['data'].get('label_format', 'binary_wide'),
    label_column_prefix=cfg['data'].get('label_column_prefix', 'Label'),
    label_code_regex=cfg['data'].get('label_code_regex'),
    is_test=False,
)
df, label_cols = train_ds.df, train_ds.label_cols
print(f'n_samples = {len(df):>5}')
print(f'n_labels  = {len(label_cols):>5}')
print(f'issues    = {issues}')
df.head(3)

## 2. Exploratory Data Analysis

We surface the four EDA artifacts cited in the report: label frequency distribution, multi-label cardinality, label co-occurrence, and document length distribution.

In [ ]:
freq = eda_mod.label_frequency_table(df, label_cols)
card = eda_mod.cardinality_stats(df, label_cols)
doc_lens = eda_mod.document_length_stats(df[train_ds.text_col])
print('Cardinality stats:', json.dumps(card, indent=2))
print('Document lengths (words):', json.dumps(doc_lens['words'], indent=2))
freq

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=freq, x='label', y='positives', ax=axes[0], color='steelblue')
axes[0].set_title('Per-label positive counts (head→tail)')
axes[0].tick_params(axis='x', rotation=60)
axes[1].hist([len(t.split()) for t in df[train_ds.text_col]], bins=40, color='steelblue')
axes[1].set(title='Document length (words)', xlabel='tokens', ylabel='documents')
plt.tight_layout()
Path('reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('reports/figures/eda_overview.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
cooc = eda_mod.cooccurrence_matrix(df, label_cols)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cooc, annot=False, cmap='magma', ax=ax)
ax.set_title('Label co-occurrence (joint positive counts)')
plt.tight_layout()
plt.savefig('reports/figures/cooccurrence.png', dpi=140, bbox_inches='tight')
plt.show()

## 3. Run the 8 progressive experiments

Each experiment is a separate YAML config under `configs/experiments/`. The runner returns an `ExperimentResult` containing the metric bundle and per-label breakdown. We collect them into a single comparison table that drives the Results section of the report.

**Narrative:**
1. **Exp 1** — TF-IDF + Logistic Regression baseline.
2. **Exp 2** — Preprocessing ablation (stopwords/lemma/numbers).
3. **Exp 3** — Add character n-gram TF-IDF.
4. **Exp 4** — Classifier sweep (LR / LinearSVC / ComplementNB).
5. **Exp 5** — Class imbalance remedies (sample weights, MLSMOTE).
6. **Exp 6** — Add Sentence-BERT semantic embeddings.
7. **Exp 7** — Per-label threshold tuning on validation.
8. **Exp 8** — Calibrated ensemble of best two single models.

In [ ]:
EXP_DIR = Path('configs/experiments')
results = []

def expand_and_run(cfg_path):
    cfg = load_config(cfg_path)
    # Exp 4 sweep
    if cfg['model'].get('type') == '__sweep__':
        for arm in cfg['model']['sweep']:
            c = json.loads(json.dumps(cfg)); c['model']['type'] = arm
            c['name'] = f"{cfg['name']}__{arm}"
            results.append(run_experiment(c))
        return
    # Exp 5 imbalance sweep
    if isinstance(cfg['model'].get('imbalance'), dict) and 'sweep' in cfg['model']['imbalance']:
        for arm in cfg['model']['imbalance']['sweep']:
            c = json.loads(json.dumps(cfg))
            c['model']['imbalance'] = {'active': arm}
            c['name'] = f"{cfg['name']}__{arm}"
            results.append(run_experiment(c))
        return
    results.append(run_experiment(cfg))

for path in sorted(EXP_DIR.glob('exp*.yaml')):
    expand_and_run(path)

summary = pd.DataFrame([{'experiment': r.config_name, **r.metrics.as_dict()} for r in results])
summary = summary.sort_values('hamming_loss').reset_index(drop=True)
summary.to_csv('reports/experiment_summary.csv', index=False)
summary

## 4. Results visualization

Two figures drive the Results section: (a) experiment-vs-Hamming-Loss bar with secondary F1 line, and (b) per-label F1 comparison for the baseline vs the final model.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4.5))
order = summary['experiment'].tolist()
ax1.bar(order, summary['hamming_loss'], color='#4c72b0', label='Hamming Loss (↓)')
ax1.set_ylabel('Hamming Loss'); ax1.set_ylim(0, summary['hamming_loss'].max() * 1.2)
ax1.tick_params(axis='x', rotation=70)
ax2 = ax1.twinx()
ax2.plot(order, summary['macro_f1'], color='#dd8452', marker='o', label='Macro-F1 (↑)')
ax2.set_ylabel('Macro-F1'); ax2.set_ylim(0, 1)
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc='upper right')
plt.tight_layout()
plt.savefig('reports/figures/experiment_progression.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Per-label F1: baseline (Exp 1) vs best (lowest Hamming).
by_name = {r.config_name: r for r in results}
best_name = summary.iloc[0]['experiment']
baseline_name = next(n for n in by_name if n.startswith('exp01'))
rows = []
for lc in by_name[baseline_name].label_names:
    rows.append({
        'label': lc,
        'baseline_f1': by_name[baseline_name].per_label[lc]['f1'],
        'best_f1':     by_name[best_name].per_label[lc]['f1'],
        'support':     by_name[best_name].per_label[lc]['support'],
    })
perlbl = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(perlbl))
ax.bar(x - 0.2, perlbl['baseline_f1'], width=0.4, label=f'Baseline ({baseline_name})')
ax.bar(x + 0.2, perlbl['best_f1'],     width=0.4, label=f'Best ({best_name})')
ax.set_xticks(x); ax.set_xticklabels(perlbl['label'], rotation=60)
ax.set_ylabel('F1'); ax.set_title('Per-label F1: baseline vs best')
ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/per_label_f1.png', dpi=140, bbox_inches='tight')
plt.show()
perlbl

## 5. Inference on the Devex test set & submission CSV

We refit the winning experiment on **all** training data (no hold-out), encode the test set, apply the validation-tuned thresholds, and write a submission CSV.

In [ ]:
import joblib
from sdgtext.features.tfidf import make_word_tfidf, make_char_tfidf
from sdgtext.features.embeddings import encode_sbert
from sdgtext.features.combine import stack_features
from sdgtext.models.heads import build_head, predict_proba_safe
from sdgtext.eval.metrics import apply_thresholds

WINNING_CFG = load_config('experiments/exp08_calibrated_ensemble.yaml')
pcfg = PreprocessConfig.from_dict(WINNING_CFG['preprocessing'])
texts_all = normalize_corpus(df[train_ds.text_col].tolist(), pcfg)
Y_all = df[label_cols].values.astype(int)

# Re-fit features on the full training corpus.
v_word = make_word_tfidf(WINNING_CFG['features']['tfidf_word'])
v_char = make_char_tfidf(WINNING_CFG['features']['tfidf_char'])
Xw = v_word.fit_transform(texts_all)
Xc = v_char.fit_transform(texts_all)
Xe = encode_sbert(texts_all, **WINNING_CFG['features']['sbert'])
X_all = stack_features([Xw, Xc, Xe])

head = build_head('logreg_ovr', WINNING_CFG['model'].get('logreg', {}), seed=cfg['seed'])
head.fit(X_all, Y_all)

# Thresholds come from the best-experiment artifact saved in step 3.
thr_path = Path('artifacts/models') / summary.iloc[0]['experiment'] / 'thresholds.json'
thr_dict = json.loads(thr_path.read_text())
thr = np.array([thr_dict[c] for c in label_cols])

# Encode test set.
test_ds, _ = load_devex(
    path=cfg['data']['raw_test'],
    text_columns=cfg['data']['text_columns'],
    id_column=cfg['data'].get('id_column'),
    meta_columns=cfg['data'].get('meta_columns', []),
    label_format=cfg['data'].get('label_format', 'binary_wide'),
    label_column_prefix=cfg['data'].get('label_column_prefix', 'Label'),
    label_code_regex=cfg['data'].get('label_code_regex'),
    is_test=True,
)
test_texts = normalize_corpus(test_ds.df[test_ds.text_col].tolist(), pcfg)
Xw_te = v_word.transform(test_texts)
Xc_te = v_char.transform(test_texts)
Xe_te = encode_sbert(test_texts, **WINNING_CFG['features']['sbert'])
X_te = stack_features([Xw_te, Xc_te, Xe_te])
P_te = predict_proba_safe(head, X_te)
Y_te = apply_thresholds(P_te, thr, min_labels_per_doc=WINNING_CFG['inference'].get('min_labels_per_doc', 0))

sub = pd.DataFrame(Y_te, columns=label_cols)
if test_ds.id_col and test_ds.id_col in test_ds.df.columns:
    sub.insert(0, test_ds.id_col, test_ds.df[test_ds.id_col].values)
Path('artifacts/predictions').mkdir(parents=True, exist_ok=True)
sub_path = Path('artifacts/predictions/submission.csv')
sub.to_csv(sub_path, index=False)

# Persist an inference bundle that the CLI's `predict` command can load.
bundle = {
    'features': None,         # the notebook's feature pipeline is multi-block; CLI bundle is provided separately
    'model': head,
    'thresholds': thr.tolist(),
    'label_names': list(label_cols),
}
joblib.dump(bundle, Path('artifacts/models') / summary.iloc[0]['experiment'] / 'pipeline.joblib')
print(f'Submission written: {sub_path} ({len(sub)} rows)')
sub.head()

## 6. What to put in the report

* `reports/experiment_summary.csv` — comparison table (drops directly into the Results section).
* `reports/figures/eda_overview.png`, `cooccurrence.png` — EDA & dataset section.
* `reports/figures/experiment_progression.png` — Results section primary figure.
* `reports/figures/per_label_f1.png` — head/tail analysis for Discussion.
* `artifacts/models/<exp>/metrics.json` — raw numbers per experiment (cite verbatim, don't paraphrase).

Open `docs/REPORT_TEMPLATE.md` for the academic template the team fills in.